In [23]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [24]:
SYMBOL = '1000PEPE'
INTERVAL = '1h'
SEED = 42

In [25]:
db_path = '../database/financial_data.duckdb'
conn = duckdb.connect(db_path, read_only=True)

query = f"select * from gold_ml_features where asset_symbol = '{SYMBOL}' and interval = '{INTERVAL}'"
df = conn.execute(query).df()
print(f"total rows loaded: {len(df)}")

total rows loaded: 25867


In [26]:
low = df['log_returns'].quantile(0.01)
high = df['log_returns'].quantile(0.99)
df['log_returns_cleaned'] = df['log_returns'].clip(lower=low, upper=high)

df['rsi_14_lag1'] = df['rsi_14'].shift(1)
df['log_returns_lag1'] = df['log_returns_cleaned'].shift(1)
df['volume_lag1'] = df['volume'].shift(1)

df['price_vs_sma'] = (df['close'] - df['sma_30']) / df['sma_30']
df['vol_spike'] = df['volume'] / df['volume_sma_20']
df['momentum_24h'] = df['close'].pct_change(24)

df['target'] = (df['log_returns'].shift(-1) > 0).astype(int)

df = df.dropna()
print(f"rows after cleaning: {len(df)}")
print(df['target'].value_counts())


rows after cleaning: 25843
target
0    13070
1    12773
Name: count, dtype: int64


In [27]:
features = [
    'volume', 'rsi_14', 'atr_14', 'log_returns_cleaned', 
    'rsi_14_lag1', 'log_returns_lag1', 'volume_lag1',
    'price_vs_sma', 'bb_width', 'vol_spike',
    'momentum_24h'
]

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

X = df[features]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"training: {len(X_train)} candles")
print(f"testing: {len(X_test)} candles")


training: 20674 candles
testing: 5169 candles


In [28]:
models = {
    'RandomForest': RandomForestClassifier(
        n_estimators=1000, 
        max_depth=12, 
        random_state=SEED, 
        n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=1000, 
        max_depth=4,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED, 
        n_jobs=-1,
        eval_metric='logloss'
    )
}


In [29]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment(f"{SYMBOL}_Model_Comparison_XGtuned")

results = {}

for model_name, model in models.items():
    print(f"\n{'='*50}")
    print(f"training: {model_name}")
    print(f"{'='*50}")
    
    model.fit(X_train, y_train)
    
    predictions = model.predict(X_test)
    
    acc = accuracy_score(y_test, predictions)
    prec = precision_score(y_test, predictions)
    rec = recall_score(y_test, predictions)
    
    results[model_name] = {
        'accuracy': acc, 
        'precision': prec, 
        'recall': rec
    }
    
    with mlflow.start_run(run_name=f"{SYMBOL}_{model_name}"):
        mlflow.log_param("asset", SYMBOL)
        mlflow.log_param("interval", INTERVAL)
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("n_estimators", 1000)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        
        if model_name == 'RandomForest':
            mlflow.sklearn.log_model(model, f"rf_{SYMBOL}")
        else:
            mlflow.xgboost.log_model(model, f"xgb_{SYMBOL}")
    
    print(f"accuracy: {acc:.4f}")
    print(f"precision: {prec:.4f}")
    print(f"recall: {rec:.4f}")

print(f"\n{'='*50}")
print("RESULTS:")
print(f"{'='*50}")
for name, metrics in results.items():
    print(f"{name}: {metrics['accuracy']:.4f}")


2026/04/23 13:48:06 INFO mlflow.tracking.fluent: Experiment with name '1000PEPE_Model_Comparison_XGtuned' does not exist. Creating a new experiment.



training: RandomForest


2026/04/23 13:48:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/23 13:48:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run 1000PEPE_RandomForest at: http://localhost:5000/#/experiments/8/runs/51de2fc1b48b4b91b8befd792b3cc5d5
🧪 View experiment at: http://localhost:5000/#/experiments/8
accuracy: 0.5258
precision: 0.5084
recall: 0.5445

training: XGBoost


2026/04/23 13:48:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run 1000PEPE_XGBoost at: http://localhost:5000/#/experiments/8/runs/a2acbeaec5054a799d5290acaf0708a8
🧪 View experiment at: http://localhost:5000/#/experiments/8
accuracy: 0.5247
precision: 0.5073
recall: 0.5437

RESULTS:
RandomForest: 0.5258
XGBoost: 0.5247
